# Student Performance: Machine Learning


## Aim
Build a linear regression model to predict `Exam_Score` using all 19 available predictors and assess their predictive importance.

## Why linear regression?
Exam score is a numerical outcome, making this a regression task. Linear regression provides an interpretable starting point for examining how multiple predictors relate to exam performance.

## Plan
1. Load the cleaned dataset and check its structure.
2. Separate the 19 predictors from the target, `Exam_Score`.
3. Split the data into training and test sets.
4. Prepare numerical and categorical predictors using a preprocessing pipeline.
5. Train linear regression and compare it with a baseline prediction.
6. Evaluate performance using MAE, RMSE and R².
7. Calculate permutation importance on the test set for all 19 predictors.
8. Export the importance results for the Power BI dashboard.

## Interpretation
Predictive importance describes how much the trained model relies on each factor. It does not establish that a factor causes changes in exam scores.

In [1]:
from pathlib import Path
import pandas as pd

# Pathlib documentation: working with file and folder paths.
# https://docs.python.org/3/library/pathlib.html

# Locate the project root from either the project or notebook folder.
project_root = Path.cwd()
if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

# Load the cleaned CSV file into a pandas DataFrame.
data_path = project_root / "data" / "processed" / "StudentPerformanceFactors_cleaned.csv"
df = pd.read_csv(data_path)

# Display the number of rows and columns.
print("Dataset shape:", df.shape)

# Count missing values across all columns.
print("Remaining missing values:", df.isna().sum().sum())

# Preview the first five rows.
df.head()

Dataset shape: (6607, 20)
Remaining missing values: 0


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [3]:
# Reference: pandas documentation for DataFrame.drop:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html
# Code generated with ChatGPT assistance and adapted for this project.
# X contains all 19 predictors; y contains the exam score to predict.
# Help: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html
X = df.drop(columns=["Exam_Score"])
y = df["Exam_Score"]

print("Predictors:", X.shape)
print("Target:", y.shape)

Predictors: (6607, 19)
Target: (6607,)


## Training and Test Split

The dataset was randomly split into two sets:

- **Training set:** 5,285 rows (80%), used to fit the model.
- **Test set:** 1,322 rows (20%), reserved for evaluating predictions on unseen data.

Both sets contain all 19 predictors. `Exam_Score` is the target variable.

Setting `random_state=42` makes the split reproducible. Preprocessing will be fitted on the training set and then applied to the test set.

In [4]:
# Reference: scikit-learn documentation for train_test_split:
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
# Code generated with ChatGPT assistance and adapted for this project.

from sklearn.model_selection import train_test_split

# Reserve 20% of the data for testing and use 80% for training.
# random_state=42 makes the split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (5285, 19)
Test set: (1322, 19)


In [5]:
# Reference: pandas documentation for DataFrame.select_dtypes:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html
# Code generated with ChatGPT assistance and adapted for this project.

# Identify numerical and categorical predictors in the training data.
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numerical predictors:", len(numeric_features))
print(numeric_features)

print("\nCategorical predictors:", len(categorical_features))
print(categorical_features)

Numerical predictors: 6
['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']

Categorical predictors: 13
['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


## Predictor Types

The model uses six numerical predictors and thirteen categorical predictors.

- **Numerical predictors** will be standardised to put them on a common scale.
- **Categorical predictors** will be one-hot encoded into numerical indicator columns. This avoids assuming equal spacing between categories such as Low, Medium and High.

Scaling and encoding will be fitted using only the training data, then applied to the test data through a preprocessing pipeline.

The final importance analysis will report results for the original 19 predictors, grouping each categorical variable's encoded columns together.

In [6]:
# References: scikit-learn preprocessing documentation:
# https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
# Code generated with ChatGPT assistance and adapted for this project.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define how each predictor type will be prepared.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("Preprocessor created. It has not yet been fitted.")

Preprocessor created. It has not yet been fitted.


In [7]:
# References: scikit-learn documentation:
# https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html
# Code generated with ChatGPT assistance and adapted for this project.

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Combine preprocessing and linear regression in one pipeline.
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

# Fit preprocessing and the model using only the training data.
model.fit(X_train, y_train)

print("Linear regression model trained successfully.")

Linear regression model trained successfully.


## Model Training

A linear regression model was trained using all 19 predictors through a preprocessing pipeline.

The pipeline learns numerical scaling parameters and categorical encodings from the training data, then fits the regression model. The test set is not used during fitting.

Training is complete; predictive performance will be assessed next against a baseline model.